In [ ]:
from pathlib import Path
import sys
import matplotlib as mpl
CAMERA_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'figure_generation_notebooks' else Path.cwd().resolve()
EXPERIMENT_ROOT = Path('/mnt/e/watermark_rev2')
FIGURETYPE1_DIR = CAMERA_ROOT / 'figuretype1'
FIGURETYPE1_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EXPERIMENT_ROOT))
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import seaborn as sns
import pandas as pd
import os
import sys
sys.path.insert(0, str(EXPERIMENT_ROOT))
from theorytools import power, deltaP, KL_theory
FIGURETYPE1_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 120


In [ ]:
DELTA_VALUES = [2]
GAMMA_MIN = 0.01
GAMMA_MAX = 0.99
GAMMA_POINTS = 1000
ALPHA = 0.05
N_VALUES = [30, 50, 100, 200]


In [ ]:
gammas = np.linspace(GAMMA_MIN, GAMMA_MAX, GAMMA_POINTS)

def calculate_power_curve(gammas, delta, n, alpha):
    powers = []
    for g in gammas:
        try:
            p = power(g, delta, alpha, n)
            powers.append(p)
        except:
            powers.append(np.nan)
    return np.array(powers)
results = {}
for delta in DELTA_VALUES:
    results[delta] = {}
    for n in N_VALUES:
        results[delta][n] = calculate_power_curve(gammas, delta, n, ALPHA)


In [ ]:
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from scipy.stats import norm

def set_paper_style(font_scale=1.2):
    sns.set_theme(style='white')
    sns.set_context('paper', font_scale=font_scale)
    plt.rcParams.update({'axes.grid': False, 'grid.alpha': 0.0, 'axes.edgecolor': 'black', 'axes.linewidth': 1.0, 'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white'})

def style_axes_clean(ax):
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for side in ['left', 'bottom']:
        ax.spines[side].set_color('black')
        ax.spines[side].set_linewidth(1.0)
    ax.tick_params(axis='both', direction='out', length=3, width=1)

def plot_power_vs_gamma_each_n(N_VALUES, DELTA_VALUES, results, gammas, ALPHA, out_dir=str(FIGURETYPE1_DIR), font_scale=1.25):
    set_paper_style(font_scale)
    os.makedirs(out_dir, exist_ok=True)
    from matplotlib.colors import LinearSegmentedColormap
    norm_delta = Normalize(vmin=float(min(DELTA_VALUES)), vmax=float(max(DELTA_VALUES)))
    sns_colors = sns.color_palette('muted', as_cmap=False)
    cmap = LinearSegmentedColormap.from_list('sns_muted', sns_colors)
    sm = ScalarMappable(norm=norm_delta, cmap=cmap)
    for n in N_VALUES:
        fig, ax = plt.subplots(figsize=(6.5, 4.2), facecolor='white')
        style_axes_clean(ax)
        for delta in DELTA_VALUES:
            power_values = results[delta][n]
            ax.plot(gammas, power_values, color=cmap(norm_delta(delta)), linewidth=1.6, alpha=0.85)
        gamma_star = n / (norm.ppf(1 - ALPHA) ** 2 + n)
        ax.axvline(gamma_star, color='#1b7837', linewidth=1.2, alpha=0.7)
        ax.axhline(0.5, color='#b2182b', linestyle='--', linewidth=1.0, alpha=0.35)
        ax.axhline(0.95, color='#2166ac', linestyle='--', linewidth=1.0, alpha=0.35)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
        ax.set_xlabel('$\\gamma$')
        ax.set_ylabel('Detection Power')
        ax.set_title(f'$n={n}$, $\\alpha={ALPHA}$', fontsize=11)
        gamma_handle = Line2D([], [], linestyle='None', marker='|', markersize=16, markeredgewidth=2.0, color='#1b7837', label='$\\gamma^*=\\frac{n}{z_{1-\\alpha}^2+n}$')
        aux_handles = [gamma_handle, Line2D([0], [0], color='#b2182b', linestyle='--', linewidth=1.3, label='Power = 0.5'), Line2D([0], [0], color='#2166ac', linestyle='--', linewidth=1.3, label='Power = 0.95')]
        leg = ax.legend(handles=aux_handles, loc='upper left', fontsize=9, frameon=True)
        leg.get_frame().set_facecolor('white')
        leg.get_frame().set_edgecolor('#cccccc')
        leg.get_frame().set_linewidth(0.8)
        cbar = fig.colorbar(sm, ax=ax, pad=0.02)
        cbar.set_label('$\\delta$', fontsize=10)
        cbar.ax.tick_params(labelsize=9)
        fname = f'{out_dir}/power_vs_gamma_n_{n}.pdf'
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close(fig)


In [ ]:
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import norm
from scipy.interpolate import interp1d
from theorytools import KL_theory

def set_paper_style(font_scale=1.2):
    sns.set_theme(style='white')
    sns.set_context('paper', font_scale=font_scale)
    plt.rcParams.update({'axes.grid': False, 'axes.edgecolor': 'black', 'axes.linewidth': 1.0, 'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white'})

def style_axes_clean(ax):
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for side in ['left', 'bottom']:
        ax.spines[side].set_linewidth(1.0)
        ax.spines[side].set_color('black')
    ax.tick_params(axis='both', length=0, colors='black')

def gamma0(delta):
    delta = float(delta)
    ed = np.exp(delta)
    denom = (ed - 1.0) ** 2
    if denom == 0:
        return np.nan
    return (delta * ed - (ed - 1.0)) / denom

def find_crossings(x, y, level):
    crossings = []
    for i in range(len(y) - 1):
        if y[i] <= level < y[i + 1] or y[i] >= level > y[i + 1]:
            t = (level - y[i]) / (y[i + 1] - y[i])
            x_cross = x[i] + t * (x[i + 1] - x[i])
            crossings.append(x_cross)
    return crossings

def plot_power_vs_gamma_each_n(N_VALUES, DELTA_VALUES, results, gammas, ALPHA, out_dir=str(FIGURETYPE1_DIR), font_scale=1.25):
    set_paper_style(font_scale)
    os.makedirs(out_dir, exist_ok=True)
    base_palette = list({'grid': '#1f77b4', 'klt': '#d62728', 'opt': '#9467bd', 'dip': '#ff7f0e', 'new': '#8c564b'}.values())
    palette = [base_palette[i % len(base_palette)] for i in range(len(DELTA_VALUES))]
    gammas = np.asarray(gammas)
    for n in N_VALUES:
        fig, ax = plt.subplots(figsize=(6.5, 4.2), facecolor='white')
        style_axes_clean(ax)
        ax2 = ax.twinx()
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_linewidth(1.0)
        ax2.spines['right'].set_color('black')
        ax2.tick_params(axis='y', length=0, colors='black')
        ax.set_zorder(ax2.get_zorder() + 1)
        ax.patch.set_visible(False)
        for i, delta in enumerate(DELTA_VALUES):
            power_values = np.asarray(results[delta][n])
            ax.plot(gammas, power_values, color=palette[i], linewidth=1.7, linestyle='--', alpha=0.95)
            kl_values = np.array([KL_theory(g, delta) for g in gammas])
            ax2.plot(gammas, kl_values, color=palette[i], linewidth=2.0, linestyle='-', alpha=0.9)
        gamma_star = n / (norm.ppf(1 - ALPHA) ** 2 + n)
        ax.axvline(gamma_star, color='#1b7837', linestyle='-.', linewidth=1.2, alpha=0.7, zorder=0)
        ax.text(gamma_star, 1.01, '$\\gamma^*$', fontsize=8, color='#1b7837', ha='center', va='bottom')
        ax.axhline(0.5, color='#7570b3', linestyle='--', linewidth=1.0, alpha=0.6, zorder=0)
        ax.axhline(0.95, color='#2e7d32', linestyle='--', linewidth=1.0, alpha=0.6, zorder=0)
        ax.text(0.98, 0.5, '0.5', fontsize=8, color='#7570b3', ha='right', va='bottom', transform=ax.get_yaxis_transform())
        ax.text(0.98, 0.95, '0.95', fontsize=8, color='#2e7d32', ha='right', va='bottom', transform=ax.get_yaxis_transform())
        ax.set_xlim(0, 1.0)
        ax.set_ylim(0, 1.02)
        ax.set_xlabel('$\\gamma$')
        ax.set_ylabel('Detection Power')
        ax2.set_ylabel('KL Divergence', color='gray')
        ax2.tick_params(axis='y', labelcolor='gray')
        legend_handles = [Line2D([0], [0], color=palette[0], linestyle='--', lw=1.7, label='Power'), Line2D([0], [0], color=palette[0], linestyle='-', lw=2.0, label='KL')]
        leg = ax.legend(handles=legend_handles, loc='lower left', fontsize=8, frameon=True, ncol=2, handlelength=1.8, columnspacing=1.0, handletextpad=0.5, framealpha=1.0)
        leg.get_frame().set_facecolor('white')
        leg.get_frame().set_edgecolor('#cccccc')
        leg.get_frame().set_linewidth(0.8)
        leg.get_frame().set_alpha(1.0)
        fname = f'{out_dir}/power_vs_gamma_n_{n}.pdf'
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close(fig)


In [ ]:
plot_power_vs_gamma_each_n(N_VALUES=N_VALUES, DELTA_VALUES=DELTA_VALUES, results=results, gammas=gammas, ALPHA=ALPHA)


In [ ]:
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import norm
from scipy.optimize import brentq
from theorytools import KL_theory, power

def solve_kl_roots(c, delta, tol=1e-09):
    ed = np.exp(delta)
    denom = (ed - 1.0) ** 2
    g0 = (delta * ed - (ed - 1.0)) / denom
    kl_max = KL_theory(g0, delta)
    if c > kl_max or c <= 0:
        return (None, None)

    def f(g):
        return KL_theory(g, delta) - c
    try:
        gamma_small = brentq(f, 1e-06, g0 - tol)
    except:
        gamma_small = None
    try:
        gamma_large = brentq(f, g0 + tol, 1 - 1e-06)
    except:
        gamma_large = None
    return (gamma_small, gamma_large)

def plot_kl_power_comparison(delta=2, n=50, alpha=0.05, n_points=100, out_dir=str(FIGURETYPE1_DIR), font_scale=1.25):
    sns.set_theme(style='white')
    sns.set_context('paper', font_scale=font_scale)
    plt.rcParams.update({'axes.grid': False, 'axes.edgecolor': 'black', 'axes.linewidth': 1.0, 'figure.facecolor': 'white', 'axes.facecolor': 'white'})
    os.makedirs(out_dir, exist_ok=True)
    ed = np.exp(delta)
    g0 = (delta * ed - (ed - 1.0)) / (ed - 1.0) ** 2
    kl_max = KL_theory(g0, delta)
    kl_values = np.linspace(kl_max * 0.99, 0.001, n_points)
    colors = ['#1f77b4', '#d62728']
    fig, ax = plt.subplots(figsize=(7, 4.5), facecolor='white')
    power_small = []
    power_large = []
    valid_kl = []
    for kl in kl_values:
        g_small, g_large = solve_kl_roots(kl, delta)
        if g_small is not None and g_large is not None:
            p_small = power(g_small, delta, alpha, n)
            p_large = power(g_large, delta, alpha, n)
            power_small.append(p_small)
            power_large.append(p_large)
            valid_kl.append(kl)
    valid_kl = np.array(valid_kl)
    power_small = np.array(power_small)
    power_large = np.array(power_large)
    ax.plot(valid_kl, power_large, color=colors[0], linestyle='-', linewidth=2.0, alpha=0.9, label=f'$\\gamma_{{high}}$')
    ax.plot(valid_kl, power_small, color=colors[1], linestyle='-', linewidth=2.0, alpha=0.9, label=f'$\\gamma_{{low}}$')
    p_at_g0 = power(g0, delta, alpha, n)
    ax.scatter([kl_max], [p_at_g0], s=30, color='black', zorder=10)
    ax.text(kl_max * 1.02, p_at_g0 + 0.01, f'$\\gamma_0(\\delta)$', fontsize=9, ha='left', va='bottom')
    ax.set_xlim(kl_max * 1.05, 0)
    ax.set_ylim(0.6, 1.02)
    ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
    ax.set_xlabel('KL Divergence')
    ax.set_ylabel('Detection Power')
    ax.tick_params(axis='both', length=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    leg = ax.legend(loc='lower left', fontsize=9, frameon=True)
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_edgecolor('#cccccc')
    leg.get_frame().set_alpha(1.0)
    fname = f'{out_dir}/kl_power_comparison_delta_{delta}_n_{n}.pdf'
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
plot_kl_power_comparison(delta=2, n=50, alpha=0.05)


In [ ]:
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from theorytools import KL_theory, power

def find_power_crossing(delta, n, alpha=0.05, tol=1e-09):
    ed = np.exp(delta)
    denom = (ed - 1.0) ** 2
    g0 = (delta * ed - (ed - 1.0)) / denom
    kl_max = KL_theory(g0, delta)

    def power_diff(kl):
        if kl <= 0 or kl >= kl_max:
            return 0

        def f(g):
            return KL_theory(g, delta) - kl
        try:
            g_small = brentq(f, 1e-06, g0 - tol)
            g_large = brentq(f, g0 + tol, 1 - 1e-06)
        except:
            return 0
        p_small = power(g_small, delta, alpha, n)
        p_large = power(g_large, delta, alpha, n)
        return p_large - p_small
    kl_samples = np.linspace(0.001, kl_max * 0.99, 200)
    diffs = [power_diff(kl) for kl in kl_samples]
    crossing_kl = None
    crossing_power = None
    for i in range(len(diffs) - 1):
        if diffs[i] * diffs[i + 1] < 0:
            try:
                crossing_kl = brentq(power_diff, kl_samples[i], kl_samples[i + 1])

                def f(g):
                    return KL_theory(g, delta) - crossing_kl
                g_small = brentq(f, 1e-06, g0 - tol)
                crossing_power = power(g_small, delta, alpha, n)
                break
            except:
                pass
    return (crossing_kl, crossing_power)

def plot_crossing_power_vs_delta(n=50, alpha=0.05, delta_range=(0.5, 4.0), n_points=50, out_dir=str(FIGURETYPE1_DIR), font_scale=1.25):
    sns.set_theme(style='white')
    sns.set_context('paper', font_scale=font_scale)
    plt.rcParams.update({'axes.grid': False, 'axes.edgecolor': 'black', 'axes.linewidth': 1.0, 'figure.facecolor': 'white', 'axes.facecolor': 'white'})
    os.makedirs(out_dir, exist_ok=True)
    deltas = np.linspace(delta_range[0], delta_range[1], n_points)
    crossing_powers = []
    valid_deltas = []
    for delta in deltas:
        kl, pwr = find_power_crossing(delta, n, alpha)
        if kl is not None and pwr is not None:
            crossing_powers.append(pwr)
            valid_deltas.append(delta)
    valid_deltas = np.array(valid_deltas)
    crossing_powers = np.array(crossing_powers)
    fig, ax = plt.subplots(figsize=(6, 4.5), facecolor='white')
    ax.plot(valid_deltas, crossing_powers, color='#1f77b4', linewidth=2.0)
    ax.set_xlabel('$\\delta$')
    ax.set_ylabel('Intersection Power')
    ax.tick_params(axis='both', length=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    fname = f'{out_dir}/crossing_power_vs_delta_n_{n}.pdf'
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    return (valid_deltas, crossing_powers)
plot_crossing_power_vs_delta(n=50, alpha=0.05, delta_range=(0.1, 8.0))


In [ ]:
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import norm
from scipy.optimize import brentq
from theorytools import KL_theory, power

def plot_combined_three_panel(delta=2, n=50, alpha=0.05, delta_range=(0.1, 8.0), n_points=100, gammas=None, results=None, out_dir=str(FIGURETYPE1_DIR), font_scale=1.25, figsize=(6, 10), height_ratios=(1, 1, 1), yticks_num=(5, 5, 5), legend_fontsize=10, label_fontsize=12):
    sns.set_theme(style='white')
    sns.set_context('paper', font_scale=font_scale)
    plt.rcParams.update({'axes.grid': False, 'axes.edgecolor': 'black', 'axes.linewidth': 1.0, 'figure.facecolor': 'white', 'axes.facecolor': 'white'})
    os.makedirs(out_dir, exist_ok=True)
    fig, axes = plt.subplots(3, 1, figsize=figsize, facecolor='white', gridspec_kw={'height_ratios': height_ratios})
    ax_a, ax_b, ax_c = axes
    colors = ['#1f77b4', '#d62728']
    marker_color = '#00dbf8'

    def solve_kl_roots(c, delta, tol=1e-09):
        ed = np.exp(delta)
        denom = (ed - 1.0) ** 2
        g0 = (delta * ed - (ed - 1.0)) / denom
        kl_max = KL_theory(g0, delta)
        if c > kl_max or c <= 0:
            return (None, None)

        def f(g):
            return KL_theory(g, delta) - c
        try:
            gamma_small = brentq(f, 1e-06, g0 - tol)
        except:
            gamma_small = None
        try:
            gamma_large = brentq(f, g0 + tol, 1 - 1e-06)
        except:
            gamma_large = None
        return (gamma_small, gamma_large)

    def find_power_crossing(delta, n, alpha=0.05, tol=1e-09):
        ed = np.exp(delta)
        denom = (ed - 1.0) ** 2
        g0 = (delta * ed - (ed - 1.0)) / denom
        kl_max = KL_theory(g0, delta)

        def power_diff(kl):
            if kl <= 0 or kl >= kl_max:
                return 0

            def f(g):
                return KL_theory(g, delta) - kl
            try:
                g_small = brentq(f, 1e-06, g0 - tol)
                g_large = brentq(f, g0 + tol, 1 - 1e-06)
            except:
                return 0
            p_small = power(g_small, delta, alpha, n)
            p_large = power(g_large, delta, alpha, n)
            return p_large - p_small
        kl_samples = np.linspace(0.001, kl_max * 0.99, 200)
        diffs = [power_diff(kl) for kl in kl_samples]
        crossing_kl = None
        crossing_power = None
        for i in range(len(diffs) - 1):
            if diffs[i] * diffs[i + 1] < 0:
                try:
                    crossing_kl = brentq(power_diff, kl_samples[i], kl_samples[i + 1])

                    def f(g):
                        return KL_theory(g, delta) - crossing_kl
                    g_small = brentq(f, 1e-06, g0 - tol)
                    crossing_power = power(g_small, delta, alpha, n)
                    break
                except:
                    pass
        return (crossing_kl, crossing_power)

    def style_ax(ax):
        ax.tick_params(axis='both', length=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    ed = np.exp(delta)
    g0 = (delta * ed - (ed - 1.0)) / (ed - 1.0) ** 2
    kl_max = KL_theory(g0, delta)
    kl_range = np.linspace(kl_max * 0.99, 0.001, n_points)
    power_small = []
    power_large = []
    valid_kl = []
    for kl in kl_range:
        g_small, g_large = solve_kl_roots(kl, delta)
        if g_small is not None and g_large is not None:
            p_small = power(g_small, delta, alpha, n)
            p_large = power(g_large, delta, alpha, n)
            power_small.append(p_small)
            power_large.append(p_large)
            valid_kl.append(kl)
    valid_kl = np.array(valid_kl)
    power_small = np.array(power_small)
    power_large = np.array(power_large)
    ax_a.plot(valid_kl, power_large, color=colors[0], linestyle='-', linewidth=2.0, alpha=0.9, label=f'$\\gamma_{{h}}$')
    ax_a.plot(valid_kl, power_small, color=colors[1], linestyle='-', linewidth=2.0, alpha=0.9, label=f'$\\gamma_{{l}}$')
    p_at_g0 = power(g0, delta, alpha, n)
    ax_a.text(kl_max * 1.02, p_at_g0 - 0.01, f'$\\gamma_0(\\delta)$', fontsize=label_fontsize, ha='left', va='top')
    crossing_kl, crossing_power = find_power_crossing(delta, n, alpha)
    if crossing_kl is not None and crossing_power is not None:
        ax_a.scatter([crossing_kl], [crossing_power], s=30, color=marker_color, zorder=10)
        ax_a.text(crossing_kl + 0.002, crossing_power + 0.015, 'Intersection', fontsize=label_fontsize, ha='right', va='top')
    ax_a.set_xlim(kl_max * 1.05, 0)
    ax_a.set_ylim(0.6, 1.02)
    ax_a.set_yticks(np.linspace(0.6, 1.0, yticks_num[0]))
    ax_a.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))
    ax_a.set_xlabel('KL Divergence', fontsize=label_fontsize)
    ax_a.set_ylabel('Power', fontsize=label_fontsize)
    style_ax(ax_a)
    leg_a = ax_a.legend(loc='lower left', fontsize=legend_fontsize, frameon=True)
    leg_a.get_frame().set_facecolor('white')
    leg_a.get_frame().set_edgecolor('#cccccc')
    deltas = np.linspace(delta_range[0], delta_range[1], n_points)
    crossing_powers = []
    valid_deltas = []
    for d in deltas:
        kl, pwr = find_power_crossing(d, n, alpha)
        if kl is not None and pwr is not None:
            crossing_powers.append(pwr)
            valid_deltas.append(d)
    valid_deltas = np.array(valid_deltas)
    crossing_powers = np.array(crossing_powers)
    ax_b.plot(valid_deltas, crossing_powers, color='#1f77b4', linewidth=2.0, label='Power at Intersection')
    ax_b.set_xlabel('$\\delta$', fontsize=label_fontsize)
    ax_b.set_ylabel('Power', fontsize=label_fontsize)
    ymin_b, ymax_b = ax_b.get_ylim()
    ax_b.set_yticks(np.linspace(ymin_b, ymax_b, yticks_num[1]))
    ax_b.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))
    style_ax(ax_b)
    leg_b = ax_b.legend(loc='lower left', fontsize=legend_fontsize, frameon=True)
    leg_b.get_frame().set_facecolor('white')
    leg_b.get_frame().set_edgecolor('#cccccc')
    if gammas is None:
        gammas = np.linspace(0.01, 0.99, 1000)
    gammas = np.asarray(gammas)
    power_values = np.array([power(g, delta, alpha, n) for g in gammas])
    kl_values = np.array([KL_theory(g, delta) for g in gammas])
    ax_c.plot(gammas, power_values, color=colors[0], linewidth=1.7, linestyle='--', alpha=0.95)
    ax_c2 = ax_c.twinx()
    ax_c2.plot(gammas, kl_values, color=colors[0], linewidth=2.0, linestyle='-', alpha=0.9)
    ax_c2.spines['top'].set_visible(False)
    ax_c2.spines['right'].set_linewidth(1.0)
    ax_c2.tick_params(axis='y', length=0, labelcolor='gray')
    ax_c2.set_ylabel('KL Divergence', color='gray', fontsize=label_fontsize)
    ax_c.set_zorder(ax_c2.get_zorder() + 1)
    ax_c.patch.set_visible(False)
    gamma_star = n / (norm.ppf(1 - alpha) ** 2 + n)
    ax_c.axvline(gamma_star, color='#1b7837', linestyle='-.', linewidth=1.2, alpha=0.7, zorder=0)
    ax_c.text(gamma_star, 1.01, '$\\gamma^*$', fontsize=8, color='#1b7837', ha='center', va='bottom')
    ax_c.axhline(0.5, color='#7570b3', linestyle='--', linewidth=1.0, alpha=0.6, zorder=0)
    ax_c.axhline(0.95, color='#2e7d32', linestyle='--', linewidth=1.0, alpha=0.6, zorder=0)
    ax_c.text(0.98, 0.5, '0.5', fontsize=8, color='#7570b3', ha='right', va='bottom', transform=ax_c.get_yaxis_transform())
    ax_c.text(0.98, 0.95, '0.95', fontsize=8, color='#2e7d32', ha='right', va='bottom', transform=ax_c.get_yaxis_transform())
    ax_c.set_xlim(0, 1.0)
    ax_c.set_ylim(0, 1.02)
    ax_c.set_yticks(np.linspace(0, 1.0, yticks_num[2]))
    ax_c.set_xlabel('$\\gamma$', fontsize=label_fontsize)
    ax_c.set_ylabel('Detection Power', fontsize=label_fontsize)
    style_ax(ax_c)
    legend_handles_c = [Line2D([0], [0], color=colors[0], linestyle='--', lw=1.7, label='Power'), Line2D([0], [0], color=colors[0], linestyle='-', lw=2.0, label='KL')]
    leg_c = ax_c.legend(handles=legend_handles_c, loc='lower left', fontsize=legend_fontsize - 1, frameon=True, ncol=2, handlelength=1.8)
    leg_c.get_frame().set_facecolor('white')
    leg_c.get_frame().set_edgecolor('#cccccc')
    for ax, label in zip([ax_a, ax_b, ax_c], ['(a)', '(b)', '(c)']):
        ax.text(-0.08, 1.05, label, transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom', ha='left')
    plt.tight_layout()
    fname = f'{out_dir}/combined_three_panel_delta_{delta}_n_{n}.pdf'
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
plot_combined_three_panel(delta=2, n=50, alpha=0.05, delta_range=(0.1, 8.0), gammas=gammas, figsize=(6, 6), height_ratios=(1, 1, 2), yticks_num=(3, 3, 4), legend_fontsize=10, label_fontsize=12)


In [ ]:
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import norm
from scipy.optimize import brentq
from theorytools import KL_theory, power

def plot_combined_two_panel_sns(delta=2, n=50, alpha=0.05, delta_range=(0.1, 8.0), n_points=100, gammas=None, out_dir=str(FIGURETYPE1_DIR), figsize=(6, 6), height_ratios=(1, 1), yticks_num=(5, 5), palette='muted'):
    sns.set_theme(style='ticks', context='paper', font_scale=1.3)
    plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.edgecolor': '#333333', 'axes.linewidth': 1.2, 'axes.labelcolor': '#333333', 'xtick.color': '#333333', 'ytick.color': '#333333', 'text.color': '#333333', 'font.family': 'sans-serif'})
    colors = sns.color_palette(palette)
    color_main = colors[0]
    color_alt = colors[3]
    color_accent = colors[2]
    os.makedirs(out_dir, exist_ok=True)
    fig, axes = plt.subplots(2, 1, figsize=figsize, facecolor='white', gridspec_kw={'height_ratios': height_ratios})
    ax_a, ax_b = axes

    def solve_kl_roots(c, delta, tol=1e-09):
        ed = np.exp(delta)
        denom = (ed - 1.0) ** 2
        g0 = (delta * ed - (ed - 1.0)) / denom
        kl_max = KL_theory(g0, delta)
        if c > kl_max or c <= 0:
            return (None, None)

        def f(g):
            return KL_theory(g, delta) - c
        try:
            gamma_small = brentq(f, 1e-06, g0 - tol)
        except:
            gamma_small = None
        try:
            gamma_large = brentq(f, g0 + tol, 1 - 1e-06)
        except:
            gamma_large = None
        return (gamma_small, gamma_large)

    def find_power_crossing(delta, n, alpha=0.05, tol=1e-09):
        ed = np.exp(delta)
        denom = (ed - 1.0) ** 2
        g0 = (delta * ed - (ed - 1.0)) / denom
        kl_max = KL_theory(g0, delta)

        def power_diff(kl):
            if kl <= 0 or kl >= kl_max:
                return 0

            def f(g):
                return KL_theory(g, delta) - kl
            try:
                g_small = brentq(f, 1e-06, g0 - tol)
                g_large = brentq(f, g0 + tol, 1 - 1e-06)
            except:
                return 0
            p_small = power(g_small, delta, alpha, n)
            p_large = power(g_large, delta, alpha, n)
            return p_large - p_small
        kl_samples = np.linspace(0.001, kl_max * 0.99, 200)
        diffs = [power_diff(kl) for kl in kl_samples]
        crossing_kl = None
        crossing_power = None
        for i in range(len(diffs) - 1):
            if diffs[i] * diffs[i + 1] < 0:
                try:
                    crossing_kl = brentq(power_diff, kl_samples[i], kl_samples[i + 1])

                    def f(g):
                        return KL_theory(g, delta) - crossing_kl
                    g_small = brentq(f, 1e-06, g0 - tol)
                    crossing_power = power(g_small, delta, alpha, n)
                    break
                except:
                    pass
        return (crossing_kl, crossing_power)
    ed = np.exp(delta)
    g0 = (delta * ed - (ed - 1.0)) / (ed - 1.0) ** 2
    kl_max = KL_theory(g0, delta)
    kl_range = np.linspace(kl_max * 0.99, 0.001, n_points)
    power_small, power_large, valid_kl = ([], [], [])
    for kl in kl_range:
        g_small, g_large = solve_kl_roots(kl, delta)
        if g_small is not None and g_large is not None:
            power_small.append(power(g_small, delta, alpha, n))
            power_large.append(power(g_large, delta, alpha, n))
            valid_kl.append(kl)
    valid_kl = np.array(valid_kl)
    power_small = np.array(power_small)
    power_large = np.array(power_large)
    ax_a.fill_between(valid_kl, power_small, power_large, where=power_large >= power_small, alpha=0.15, color=color_main, interpolate=True)
    ax_a.fill_between(valid_kl, power_small, power_large, where=power_small > power_large, alpha=0.15, color=color_alt, interpolate=True)
    ax_a.plot(valid_kl, power_large, color=color_main, linestyle='-', linewidth=2.2, label=f'$\\gamma_{{h}}$')
    ax_a.plot(valid_kl, power_small, color=color_alt, linestyle='-', linewidth=2.2, label=f'$\\gamma_{{l}}$')
    p_at_g0 = power(g0, delta, alpha, n)
    ax_a.text(kl_max * 1.02, p_at_g0 - 0.01, f'$\\gamma_0(\\delta)$', fontsize=11, ha='left', va='top', color='#555555')
    crossing_kl, crossing_power = find_power_crossing(delta, n, alpha)
    if crossing_kl is not None:
        ax_a.scatter([crossing_kl], [crossing_power], s=50, color=color_accent, edgecolor='white', linewidth=1.5, zorder=10)
        ax_a.text(crossing_kl - 0.002, crossing_power - 0.03, 'Intersection', fontsize=10, ha='right', va='top', color='#555555')
    ax_a.set_xlim(kl_max * 1.05, 0)
    ax_a.set_ylim(0.6, 1.02)
    ax_a.set_yticks(np.linspace(0.6, 1.0, yticks_num[0]))
    ax_a.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))
    ax_a.set_xlabel('KL Divergence')
    ax_a.set_ylabel('Power')
    leg_a = ax_a.legend(loc='lower left', frameon=True, fancybox=True, shadow=False, framealpha=0.95)
    leg_a.get_frame().set_edgecolor('#cccccc')
    sns.despine(ax=ax_a)
    deltas = np.linspace(delta_range[0], delta_range[1], n_points)
    crossing_powers, valid_deltas = ([], [])
    for d in deltas:
        kl, pwr = find_power_crossing(d, n, alpha)
        if kl is not None and pwr is not None:
            crossing_powers.append(pwr)
            valid_deltas.append(d)
    valid_deltas = np.array(valid_deltas)
    crossing_powers = np.array(crossing_powers)
    ax_b.plot(valid_deltas, crossing_powers, color=color_main, linewidth=2.2, label='Power at Intersection')
    ax_b.fill_between(valid_deltas, crossing_powers, alpha=0.15, color=color_main)
    ax_b.set_xlabel('$\\delta$')
    ax_b.set_ylabel('Power')
    ax_b.set_ylim(0.7, 0.8)
    ax_b.set_yticks(np.linspace(0.7, 0.8, yticks_num[1]))
    ax_b.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))
    leg_b = ax_b.legend(loc='lower left', frameon=True, fancybox=True, framealpha=0.95)
    leg_b.get_frame().set_edgecolor('#cccccc')
    sns.despine(ax=ax_b)
    for ax, label in zip([ax_a, ax_b], ['(a)', '(b)']):
        ax.text(-0.1, 1.08, label, transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom', ha='left')
    plt.tight_layout()
    fname = f'{out_dir}/combined_two_panel_sns_delta_{delta}_n_{n}.pdf'
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
plot_combined_two_panel_sns(delta=2, n=50, alpha=0.05, delta_range=(0.1, 8.0), gammas=gammas, figsize=(6, 4), height_ratios=(1, 1), yticks_num=(3, 3), palette='muted')
